# 🎯 ODBARS Vision — YOLOv8 Model Eğitimi

**Bu notebook Colab'da GPU (T4) ile çalıştırılmalıdır.**

Çalıştırmadan önce:
1. Üst menü → **Runtime → Change runtime type → T4 GPU**
2. Roboflow projenizden **API Key** ve **Project URL** alın

---
**Sınıflar:** `tabela (0)` | `stop (1)` | `engel (2)` | `hedef (3)` | `koni (4)`

In [ ]:
# ✅ ADIM 1: GPU kontrolü
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ GPU bulunamadı! Runtime > Change runtime type > T4 GPU seçin.')

In [ ]:
# ✅ ADIM 2: Gerekli kütüphaneleri yükle
!pip install ultralytics roboflow --quiet
print('✅ Kurulum tamamlandı.')

In [ ]:
# ✅ ADIM 3: Roboflow'dan dataset indir
# Roboflow > Project > Versions > Export > YOLOv8 > Show download code
# Oradan API key ve proje adını alın.

from roboflow import Roboflow

RF_API_KEY   = "BURAYA_API_KEYINIZI_YAZIN"   # Roboflow > Settings > API
RF_WORKSPACE = "BURAYA_WORKSPACE_ADINIZI"     # Roboflow URL'sindeki workspace
RF_PROJECT   = "odbars-vision"               # Proje adı
RF_VERSION   = 1                             # Dataset versiyonu

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
version = project.version(RF_VERSION)
dataset = version.download("yolov8")

print(f'✅ Dataset indirildi: {dataset.location}')

In [ ]:
# ✅ ADIM 4: YOLOv8 modelini eğit
# yolov8n = nano (hızlı, hafif) — Jetson Nano için ideal
# yolov8s = small (biraz daha iyi doğruluk)
# Epochs: 50 başlangıç için yeterli, veri arttıkça 100'e çıkarılabilir

from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # Pretrained ağırlıkları indir (transfer learning)

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,          # Görüntü boyutu (640x640)
    batch=16,           # T4 GPU için ideal batch boyutu
    patience=10,        # 10 epoch iyileşme olmazsa erken dur
    device=0,           # GPU kullan
    project='odbars',
    name='v1',
    exist_ok=True,
    plots=True,         # Eğitim grafikleri kaydet
    save=True,
    verbose=True,
)

In [ ]:
# ✅ ADIM 5: Eğitim sonuçlarını görüntüle
from IPython.display import Image, display
import glob

# Confusion matrix ve metrik grafikleri
for img_path in glob.glob('odbars/v1/*.png'):
    print(img_path)
    display(Image(img_path, width=700))

In [ ]:
# ✅ ADIM 6: Modeli doğrula (Validation set üzerinde)
metrics = model.val()

print(f"mAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")
print(f"Precision  : {metrics.box.mp:.3f}")
print(f"Recall     : {metrics.box.mr:.3f}")

In [ ]:
# ✅ ADIM 7: Test görüntüsü üzerinde tahmin yap
# Herhangi bir test görselini /content/ altına yükleyin

test_image = '/content/test.jpg'  # Kendi test görselinizi buraya yükleyin

results = model.predict(test_image, conf=0.4, save=True, project='odbars', name='test')

for r in results:
    for box in r.boxes:
        cls_id = int(box.cls)
        conf   = float(box.conf)
        names  = ['tabela', 'stop', 'engel', 'hedef', 'koni']
        print(f"  {names[cls_id]:10s} | güven: {conf:.2f}")

display(Image('odbars/test/test.jpg', width=700))

In [ ]:
# ✅ ADIM 8: Modeli indir
# En iyi ağırlıklar odbars/v1/weights/best.pt konumunda
# Bu dosyayı indirip vision/ klasörüne koyun → main.py'de kullanılacak

from google.colab import files
files.download('odbars/v1/weights/best.pt')
print('✅ Model indirildi: best.pt')
print('   Bu dosyayı vision/ klasörüne koyun.')

## 📊 Hedef Metrikler

| Metrik | Hedef | Açıklama |
|--------|-------|----------|
| mAP50 | > 0.80 | Ana başarı kriteri |
| Precision | > 0.75 | Yanlış pozitif oranı |
| Recall | > 0.75 | Kaçırılan nesne oranı |

Sonuçlar düşükse:
- Daha fazla görüntü ekleyin (sınıf başı min. 100 önerilir)
- Roboflow augmentation'ı açın (flip, brightness, noise)
- `epochs=100` olarak artırın
- `yolov8n` yerine `yolov8s` deneyin